# 15 — Robustness and Validation

## Goal

Check whether the main results remain similar when we change reasonable assumptions.

This notebook keeps only four simple robustness checks:

1. **No-leakage checks**
2. **Random-seed stability**
3. **Target-threshold sensitivity**
4. **Short-window feature sensitivity**

Main evaluation metrics:
- Balanced Accuracy
- Macro F1

The main models are Logistic Regression, Random Forest, and XGBoost.

## 1. Setup and Load Data

We use the same dataset, target, feature exclusions, model settings, and temporal split as the pooled model.

- Train: 2020–2024
- Test: 2025
- Baseline target threshold: ±1.5%

In [1]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, f1_score
from xgboost import XGBClassifier

RANDOM_STATE = 42
TEST_YEAR = 2025

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

pd.set_option("display.float_format", "{:.4f}".format)

In [2]:
DATA_PATH = "/kaggle/input/notebooks/phyothaw/06-target-and-model-dataset-ipynb/model_ready_complete_case_2020_2025.csv"

data_df = pd.read_csv(DATA_PATH)
data_df["date"] = pd.to_datetime(data_df["date"])

# Same missing-GDELT treatment as the pooled model
data_df["gdelt_tone_missing"] = data_df["gdelt_company_tone"].isna().astype(int)
data_df["gdelt_company_tone"] = data_df["gdelt_company_tone"].fillna(0)

print("Shape:", data_df.shape)
print("Stocks:", sorted(data_df["ticker"].unique()))
print("Date range:", data_df["date"].min(), "to", data_df["date"].max())

Shape: (7370, 42)
Stocks: ['BRK-B', 'CVX', 'GE', 'MSFT', 'NVDA']
Date range: 2020-01-13 00:00:00 to 2025-12-22 00:00:00


In [3]:
TARGET = "target_5d"

DROP_COLUMNS = [
    "ticker", "date",
    "future_close_5d", "forward_return_5d",
    "movement_5d", "target_5d",
    "Dividends", "Stock Splits",
    "daily_return_check"
]

FEATURE_COLUMNS = [c for c in data_df.columns if c not in DROP_COLUMNS]

train_df = data_df[data_df["date"].dt.year < TEST_YEAR].copy()
test_df  = data_df[data_df["date"].dt.year == TEST_YEAR].copy()

X_train = train_df[FEATURE_COLUMNS]
y_train = train_df[TARGET]
X_test  = test_df[FEATURE_COLUMNS]
y_test  = test_df[TARGET]

print("Features:", len(FEATURE_COLUMNS))
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

Features: 33
Train rows: 6240
Test rows: 1130


## 2. No-Leakage Checks

These checks confirm that future information is not used as model input and that 2025 remains unseen during training.

In [4]:
forbidden = ["future_close_5d", "forward_return_5d", "movement_5d", "target_5d"]

checks = {
    "Future/target columns excluded": not any(c in FEATURE_COLUMNS for c in forbidden),
    "Training ends before testing": train_df["date"].max() < test_df["date"].min(),
    "2025 used only for testing": test_df["date"].dt.year.eq(2025).all(),
    "No NaN in model features": data_df[FEATURE_COLUMNS].isna().sum().sum() == 0
}

validation_checks = pd.DataFrame({
    "check": checks.keys(),
    "status": ["PASS" if v else "FAIL" for v in checks.values()]
})

display(validation_checks)

,check,status
0,Future/target columns excluded,PASS
1,Training ends before testing,PASS
2,2025 used only for testing,PASS
3,No NaN in model features,PASS


## 3. Baseline Models

This helper trains the same three models in every experiment so that only the tested assumption changes.

In [5]:
def run_models(train, test, features, target_col):
    X_tr, y_tr = train[features], train[target_col]
    X_te, y_te = test[features], test[target_col]

    rows = []

    # Logistic Regression
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_te_scaled = scaler.transform(X_te)

    lr = LogisticRegression(
        penalty="l2", class_weight="balanced",
        max_iter=2000, random_state=RANDOM_STATE
    )
    lr.fit(X_tr_scaled, y_tr)
    pred = lr.predict(X_te_scaled)
    rows.append(["Logistic Regression",
                 balanced_accuracy_score(y_te, pred),
                 f1_score(y_te, pred, average="macro", zero_division=0)])

    # Random Forest
    rf = RandomForestClassifier(
        n_estimators=300, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    )
    rf.fit(X_tr, y_tr)
    pred = rf.predict(X_te)
    rows.append(["Random Forest",
                 balanced_accuracy_score(y_te, pred),
                 f1_score(y_te, pred, average="macro", zero_division=0)])

    # XGBoost
    xgb = XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        objective="multi:softprob", num_class=3,
        eval_metric="mlogloss",
        random_state=RANDOM_STATE, n_jobs=-1
    )
    xgb.fit(X_tr, y_tr)
    pred = xgb.predict(X_te)
    rows.append(["XGBoost",
                 balanced_accuracy_score(y_te, pred),
                 f1_score(y_te, pred, average="macro", zero_division=0)])

    return pd.DataFrame(
        rows,
        columns=["model", "balanced_accuracy", "macro_f1"]
    )

In [6]:
baseline_results = run_models(
    train_df, test_df, FEATURE_COLUMNS, TARGET
)

display(baseline_results.sort_values("balanced_accuracy", ascending=False))

,model,balanced_accuracy,macro_f1
0,Logistic Regression,0.3899,0.3818
1,Random Forest,0.3707,0.3543
2,XGBoost,0.3526,0.3379


## 4. Random-Seed Stability

Question: **Do Random Forest and XGBoost change much when only the random seed changes?**

Five seeds are tested. Small standard deviation means the model is stable.

In [7]:
SEEDS = [42, 123, 456, 789, 2026]
seed_rows = []

for seed in SEEDS:
    rf = RandomForestClassifier(
        n_estimators=300, class_weight="balanced",
        random_state=seed, n_jobs=-1
    )
    rf.fit(X_train, y_train)
    pred = rf.predict(X_test)
    seed_rows.append([
        "Random Forest", seed,
        balanced_accuracy_score(y_test, pred),
        f1_score(y_test, pred, average="macro", zero_division=0)
    ])

    xgb = XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        objective="multi:softprob", num_class=3,
        eval_metric="mlogloss",
        random_state=seed, n_jobs=-1
    )
    xgb.fit(X_train, y_train)
    pred = xgb.predict(X_test)
    seed_rows.append([
        "XGBoost", seed,
        balanced_accuracy_score(y_test, pred),
        f1_score(y_test, pred, average="macro", zero_division=0)
    ])

seed_results = pd.DataFrame(
    seed_rows,
    columns=["model", "seed", "balanced_accuracy", "macro_f1"]
)

seed_summary = (
    seed_results.groupby("model")
    .agg(
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        balanced_accuracy_std=("balanced_accuracy", "std"),
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std")
    )
    .reset_index()
)

display(seed_summary)

,model,balanced_accuracy_mean,balanced_accuracy_std,macro_f1_mean,macro_f1_std
0,Random Forest,0.3617,0.0059,0.3338,0.0123
1,XGBoost,0.3592,0.0054,0.3415,0.0048


**Interpretation:** Small standard deviations indicate that Random Forest and XGBoost are reasonably reproducible across random seeds.

## 5. Target-Threshold Sensitivity

Baseline target: ±1.5%.

We compare:
- ±1.0%
- ±1.5%
- ±2.0%

Question: **Does changing the definition of Up/Neutral/Down change the model ranking?**

In [8]:
THRESHOLDS = {
    "1.0%": 0.010,
    "1.5%": 0.015,
    "2.0%": 0.020
}

threshold_rows = []

for name, threshold in THRESHOLDS.items():
    temp = data_df.copy()

    temp["robust_target"] = np.select(
        [
            temp["forward_return_5d"] < -threshold,
            temp["forward_return_5d"] > threshold
        ],
        [0, 2],
        default=1
    )

    train = temp[temp["date"].dt.year < TEST_YEAR].copy()
    test  = temp[temp["date"].dt.year == TEST_YEAR].copy()

    result = run_models(train, test, FEATURE_COLUMNS, "robust_target")
    result["threshold"] = name
    threshold_rows.append(result)

threshold_results = pd.concat(threshold_rows, ignore_index=True)

threshold_ba_pivot = threshold_results.pivot(
    index="threshold",
    columns="model",
    values="balanced_accuracy"
)

display(threshold_ba_pivot)

model,Logistic Regression,Random Forest,XGBoost
threshold,,,
1.0%,0.3751,0.3479,0.3498
1.5%,0.3899,0.3707,0.3526
2.0%,0.4041,0.3696,0.4007


In [9]:
best_by_threshold = (
    threshold_results.loc[
        threshold_results.groupby("threshold")["balanced_accuracy"].idxmax(),
        ["threshold", "model", "balanced_accuracy", "macro_f1"]
    ]
    .sort_values("threshold")
)

display(best_by_threshold)

,threshold,model,balanced_accuracy,macro_f1
0,1.0%,Logistic Regression,0.3751,0.3607
3,1.5%,Logistic Regression,0.3899,0.3818
6,2.0%,Logistic Regression,0.4041,0.3875


**Interpretation:** If the same model remains strongest across thresholds, the main model ranking is robust to the target definition.

## 6. Window Feature Sensitivity

Question: **Does adding short-term SMA_5 and EMA_5 features materially change performance?**

In [10]:
window_df = data_df.sort_values(["ticker", "date"]).copy()

window_df["SMA_5"] = (
    window_df.groupby("ticker")["Close"]
    .transform(lambda x: x.rolling(5).mean())
)

window_df["EMA_5"] = (
    window_df.groupby("ticker")["Close"]
    .transform(lambda x: x.ewm(span=5, adjust=False).mean())
)

window_df = window_df.dropna(subset=["SMA_5", "EMA_5"]).copy()

window_train = window_df[window_df["date"].dt.year < TEST_YEAR].copy()
window_test  = window_df[window_df["date"].dt.year == TEST_YEAR].copy()

baseline_window = run_models(
    window_train, window_test,
    FEATURE_COLUMNS, TARGET
)
baseline_window["feature_set"] = "Baseline"

extended_window = run_models(
    window_train, window_test,
    FEATURE_COLUMNS + ["SMA_5", "EMA_5"], TARGET
)
extended_window["feature_set"] = "Baseline + SMA_5 + EMA_5"

window_results = pd.concat(
    [baseline_window, extended_window],
    ignore_index=True
)

window_ba_pivot = window_results.pivot(
    index="model",
    columns="feature_set",
    values="balanced_accuracy"
)

display(window_ba_pivot)

feature_set,Baseline,Baseline + SMA_5 + EMA_5
model,,
Logistic Regression,0.3993,0.3958
Random Forest,0.3448,0.3467
XGBoost,0.3624,0.3581


**Interpretation:** Small differences between the two feature sets mean that the main conclusion is not strongly dependent on adding short-term SMA_5 and EMA_5 indicators.

## 7. Robustness Summary

The notebook tests four questions:

| Robustness check | What it asks |
|---|---|
| No leakage | Is future information excluded? |
| Random seeds | Are results reproducible? |
| Target thresholds | Does the target definition change the conclusion? |
| Short-window features | Do SMA_5/EMA_5 change the conclusion? |

Based on the completed experiments, the main results are broadly stable across these checks.

The detailed final tables and thesis-ready figures are consolidated in Notebook 16.